# LLM-as-Judge Evaluation (substitute for human evaluation)

Measures the two constructs no automatic metric reaches — **plausibility** and **educational usefulness** — and probes the **near-miss blind spot** that defeats every metric built so far (embedding similarity, numeric proximity, and exact match all score AUC ≈ 0.56 on gold vs. digit-perturbed gold).

## Validity without human labels

An LLM judge is normally validated against human ratings. Instead this validates against pairs whose correct answer is **known by construction** — the same positive-control battery that exposed Exact Match as blind (AUC 0.507, 98.2% false-negative rate).

| Control | Pair | Known answer |
|---|---|---|
| C1 | gold distractor vs random corpus distractor | gold |
| C2 | gold distractor vs the correct answer | gold |
| C3 | gold distractor vs digit-perturbed gold | gold *(probe)* |

**The script refuses to score the experiment unless the judge passes** (C1 ≥ 0.80, C2 ≥ 0.80, order-inconsistency ≤ 0.30). C3 is reported but not gated — no existing metric passes it, so it measures added capability rather than fitness.

## Bias controls

- **Judge ≠ generator family.** Qwen2.5-7B generated; Llama-3.1-8B judges. Self-preference is the most common LLM-judge failure.
- **Pairwise, not 1–5 ratings** (absolute scales drift across calls).
- **Every pair judged in both A/B orders**, so position bias is *measured* rather than assumed away. A verdict that flips with order is recorded as `inconsistent` and excluded from win rates.
- **Explicit tie option** — forcing a choice manufactures effect sizes.

Reuses the existing G1/G4/G5 generations — no regeneration. ~1 GPU-hour.

In [ ]:
# --- The ONLY cell you edit ---
REPO_URL = "https://github.com/YOUR_USERNAME/distractor.git"  # <-- set this

# Llama-3.1 is gated on HuggingFace and needs an accepted licence + token.
# If you would rather not, set JUDGE to the open alternative below.
JUDGE = "meta-llama/Llama-3.1-8B-Instruct"
# JUDGE = "mistralai/Mistral-7B-Instruct-v0.3"   # no gating, also non-Qwen
SEED = 42

In [ ]:
import torch
!nvidia-smi -L
assert torch.cuda.is_available(), "No GPU! Runtime > Change runtime type > GPU"
print("GPU OK:", torch.cuda.get_device_name(0))

In [ ]:
import os
if not os.path.exists("distractor"):
    !git clone {REPO_URL} distractor
%cd distractor
%pip install -q -r requirements_gpu.txt
if not os.path.exists("outputs/results/train_qdp.csv"):
    !python scripts/01_prepare_dataset.py
print("Ready.")

In [ ]:
# --- Upload the Experiment 2.1 generations (G1/G4/G5) ---
# Accepts EITHER exp21_results.zip OR the raw *.jsonl files.
import os, glob, shutil, zipfile
DEST = "outputs/generation/exp21"
os.makedirs(DEST, exist_ok=True)

if not os.path.exists(f"{DEST}/G4_seed42.jsonl"):
    from google.colab import files
    up = files.upload()   # select exp21_results.zip, or the G*.jsonl files
    for name in up:
        if zipfile.is_zipfile(name):
            with zipfile.ZipFile(name) as zf:
                zf.extractall(".")
            os.remove(name)
        elif name.endswith(".jsonl"):
            shutil.move(name, os.path.join(DEST, os.path.basename(name)))
        else:
            print(f"  ignoring unrecognised upload: {name}")

# Sweep up any .jsonl left in the working directory from an earlier attempt
for f in glob.glob("*.jsonl"):
    shutil.move(f, os.path.join(DEST, os.path.basename(f)))

found = sorted(os.listdir(DEST))
print(found)
assert any(f.startswith("G4_seed42.jsonl") for f in found), \
    "G4_seed42.jsonl missing — upload exp21_results.zip or the G*.jsonl files."

In [ ]:
# --- Only needed for the gated Llama model ---
# from huggingface_hub import login
# login()   # paste an HF token with Llama-3.1 access accepted

## Step 1 — Validation gate (run this alone first)

If the judge fails, **stop**. A judge that cannot separate known-good from known-bad distractors cannot support any conclusion — the same reason the Exact Match result was withdrawn.

In [ ]:
!python scripts/13_llm_judge.py --judge-model {JUDGE} --validate-only --limit 40 --seed {SEED}

## Step 2 — Full run (gate + experiment)

Runs only if the gate passes. Compares G4 vs G5, G4 vs G1, and both against the **teacher-written gold** anchor.

In [ ]:
!python scripts/13_llm_judge.py --judge-model {JUDGE} --limit 40 --exp-limit 80 --seed {SEED}

In [ ]:
print(open("outputs/results/llm_judge/llm_judge_report.md").read())

## How to read the result

**Gate fails** → the judge is unusable on this data. Report the failure; it is a legitimate finding, and it means no proxy for human evaluation is currently available. Do not use `--force`.

**Gate passes, C3 ≥ 0.70** → the judge detects near-misses that every automatic metric misses. This is the strongest outcome: you gain a capability the rest of the evaluation stack lacks.

**G4 vs G5 significant** → misconception-matched context produces distractors a validated judge can tell apart, even though the automatic metric found only a small bounded effect (+0.015, p = 0.088). That would mean the automatic metric was too insensitive, not that the effect is absent.

**G4 vs G5 not significant** → converging evidence from two independent, validated instruments that the effect is genuinely small. That is a much stronger negative claim than either alone.

**The gold anchor** gives the interpretable headline for the write-up: *"generated distractors are preferred over teacher-written ones X% of the time."* Far more meaningful than any raw score, and it bounds what the pipeline can achieve.

Always report ties and order-inconsistency alongside win rates — a high inconsistency rate undermines every verdict.

In [ ]:
!zip -qr llm_judge_results.zip outputs/results/llm_judge
from google.colab import files
files.download("llm_judge_results.zip")